In [1]:
# Librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
df = pd.read_csv('df')
df.head()

,Unnamed: 0,ID,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,...,Prima,Estudios_encoded,Estado_Civil_Divorciado,Estado_Civil_Soltero,Tipo_Jornada_Laboral_Desempleado,Tipo_Jornada_Laboral_Jornada completa,Tipo_Jornada_Laboral_Tiempo parcial,Puntos_Aleman,Puntos_Frances,Sistema_Amortizacion
0,0,KG9CT9,34,56572,73978,763,86,1,13.91,60,...,800.00,2,0,1,0,0,1,2,2,Alemán
1,1,V5O0K9,37,68698,150000,824,162,2,10.53,24,...,580.00,2,0,0,0,1,0,2,2,Alemán
2,2,YE5GD0,40,50000,7490,671,167,2,9.24,60,...,130.00,1,0,0,0,0,0,3,1,Alemán
3,3,Q6HQ3D,38,63033,11801,649,196,1,7.63,48,...,168.08,1,0,1,0,0,1,3,1,Alemán
4,4,XO8CO4,19,22301,40000,788,17,4,8.40,36,...,293.94,0,0,1,0,1,0,1,3,Francés


In [3]:
display(df.info())
display(df.describe(include="all").T)
print("Valores ausentes por columna:")
display(df.isna().sum())
print(f"Filas duplicadas: {df.duplicated().sum()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 178606 entries, 0 to 178605
Data columns (total 25 columns):
 #   Column                                 Non-Null Count   Dtype  
---  ------                                 --------------   -----  
 0   Unnamed: 0                             178606 non-null  int64  
 1   ID                                     178606 non-null  object 
 2   Edad                                   178606 non-null  int64  
 3   Ingresos                               178606 non-null  int64  
 4   Monto_Inicial                          178606 non-null  int64  
 5   Scoring_Crediticio                     178606 non-null  int64  
 6   Meses_Empleo                           178606 non-null  int64  
 7   Num_Creditos                           178606 non-null  int64  
 8   Ratio_Interes                          178606 non-null  float64
 9   Duracion                               178606 non-null  int64  
 10  Ratio_Deuda_Ingresos                   178606 non-null  

None

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Unnamed: 0,178606.0,NaN,NaN,NaN,89302.5,51559.255427,0.0,44651.25,89302.5,133953.75,178605.0
ID,178606,178606,KG9CT9,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Edad,178606.0,NaN,NaN,NaN,39.647783,10.07686,19.0,33.0,39.0,46.0,64.0
Ingresos,178606.0,NaN,NaN,NaN,41509.97074,16036.504202,16000.0,30484.25,39090.5,50096.75,120000.0
Monto_Inicial,178606.0,NaN,NaN,NaN,45190.765937,46648.083692,3000.0,15126.0,40000.0,47626.5,400000.0
Scoring_Crediticio,178606.0,NaN,NaN,NaN,692.053285,87.828224,398.0,635.0,704.0,761.0,845.0
Meses_Empleo,178606.0,NaN,NaN,NaN,212.770601,150.68807,0.0,70.0,208.0,336.0,639.0
Num_Creditos,178606.0,NaN,NaN,NaN,2.001025,1.001298,1.0,1.0,2.0,3.0,4.0
Ratio_Interes,178606.0,NaN,NaN,NaN,10.676211,3.809254,2.01,7.96,10.59,13.27,24.44
Duracion,178606.0,NaN,NaN,NaN,37.255523,14.166838,12.0,24.0,36.0,48.0,60.0


Valores ausentes por columna:


Unnamed: 0                               0
ID                                       0
Edad                                     0
Ingresos                                 0
Monto_Inicial                            0
Scoring_Crediticio                       0
Meses_Empleo                             0
Num_Creditos                             0
Ratio_Interes                            0
Duracion                                 0
Ratio_Deuda_Ingresos                     0
Posesion_Hipoteca                        0
Personas_Cargo                           0
Fiador                                   0
Impago                                   0
Prima                                    0
Estudios_encoded                         0
Estado_Civil_Divorciado                  0
Estado_Civil_Soltero                     0
Tipo_Jornada_Laboral_Desempleado         0
Tipo_Jornada_Laboral_Jornada completa    0
Tipo_Jornada_Laboral_Tiempo parcial      0
Puntos_Aleman                            0
Puntos_Fran

Filas duplicadas: 0


In [4]:
X = df.drop(columns=['Prima', 'ID', 'Puntos_Aleman', 'Puntos_Frances', "Sistema_Amortizacion"])
y = df["Prima"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [5]:
from sklearn.preprocessing import StandardScaler

# Columnas dummy (no se escalan) y el resto (sí se escalan)
dummy_cols = [c for c in X_train.columns if X_train[c].nunique() == 2]
num_cols = [c for c in X_train.columns if c not in dummy_cols]

# Creamos el escalador
scaler = StandardScaler()

# Escalamos las variables numéricas de entrenamiento
X_train_num_scal = pd.DataFrame(
    scaler.fit_transform(X_train[num_cols]),
    columns=num_cols,
    index=X_train.index
)

# Unimos las variables numéricas escaladas con las dummy originales
X_train_scaled = pd.concat(
    [X_train_num_scal, X_train[dummy_cols]], axis=1
)

# Escalamos las variables numéricas de test
X_test_num_scal = pd.DataFrame(
    scaler.transform(X_test[num_cols]),
    columns=num_cols,
    index=X_test.index
)

# Unimos las variables numéricas escaladas con las dummy originales
X_test_scaled = pd.concat(
    [X_test_num_scal, X_test[dummy_cols]], axis=1
)


In [6]:
X_train_scaled

,Unnamed: 0,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,Estudios_encoded,Posesion_Hipoteca,Personas_Cargo,Fiador,Impago,Estado_Civil_Divorciado,Estado_Civil_Soltero,Tipo_Jornada_Laboral_Desempleado,Tipo_Jornada_Laboral_Jornada completa,Tipo_Jornada_Laboral_Tiempo parcial
18397,-1.373909,0.331519,1.019653,0.058735,-2.236118,0.173239,-1.001207,-0.144744,0.757498,-0.121440,0.078533,1,1,0,0,0,0,0,1,0
54204,-0.679551,0.331519,-0.447988,-0.112445,1.308996,0.617601,-0.002893,-0.344226,0.757498,-0.397403,-0.998559,1,0,0,0,0,0,0,0,0
145820,1.097038,-1.751199,-1.542944,-0.859024,1.092414,-1.279227,-1.001207,-0.042379,-0.089539,-1.225292,-0.998559,0,0,1,0,0,1,0,1,0
148998,1.158665,-1.255314,-0.633148,-0.677434,1.081015,-1.352182,0.995422,-1.073909,-0.936576,-1.225292,0.078533,0,0,1,0,0,1,1,0,0
38113,-0.991582,0.232342,1.841103,-0.351436,0.134891,-0.058890,-0.002893,0.731925,0.757498,0.706450,1.155625,1,1,1,0,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119879,0.593998,0.629050,0.158481,-0.376969,-1.859949,0.869627,0.995422,-0.769437,-0.089539,-0.728558,-0.998559,1,1,0,0,0,0,0,1,0
103694,0.280144,-0.065189,2.802077,-0.112445,-1.232999,-0.616001,-1.001207,-0.527959,0.757498,-1.004521,2.232717,0,0,0,0,0,0,0,0,0
131932,0.827726,-1.354491,-0.653818,-0.752370,0.351473,-1.411872,-0.002893,-0.777311,1.604535,-0.176632,0.078533,0,1,0,0,0,1,0,1,0
146867,1.117341,-1.255314,-0.926203,-0.112445,0.089294,-1.411872,-0.002893,1.871071,-0.089539,-0.287017,0.078533,0,0,0,0,0,1,0,1,0


## 2. Modelo lineal 


In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_error, r2_score

# Entrenamiento del modelo
modelo_lin = LinearRegression()
modelo_lin.fit(X_train_scaled, y_train)

# Predicciones
pred_train = modelo_lin.predict(X_train_scaled)
pred_test = modelo_lin.predict(X_test_scaled)

# Métricas
for nombre, y_real, y_pred in [("Train", y_train, pred_train), ("Test", y_test, pred_test)]:
    print(f"{nombre}:  MSE = {mean_squared_error(y_real, y_pred):.2f} | "
          f"RMSE = {root_mean_squared_error(y_real, y_pred):.2f} | "
          f"MAE = {mean_absolute_error(y_real, y_pred):.2f} | "
          f"R² = {r2_score(y_real, y_pred):.3f}")

Train:  MSE = 23180.34 | RMSE = 152.25 | MAE = 120.46 | R² = 0.677
Test:  MSE = 23042.86 | RMSE = 151.80 | MAE = 120.43 | R² = 0.677


## 3. Modelo polinómico

In [8]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate, KFold

def comparar_grados(X, y, grados=(1, 2, 3)):
    """Compara modelos de regresión polinómica de distintos grados mediante validación cruzada."""

    # Definimos cómo se divide el train en la validación cruzada:
    cv = KFold(n_splits=5, shuffle=True, random_state=42)

    # Lista donde iremos guardando los resultados de cada grado
    resultados = []

    # Repetimos el mismo proceso para cada grado del polinomio
    for g in grados:
        # Generamos las variables polinómicas
        # include_bias=False evita añadir una columna de unos, porque LinearRegression ya calcula el intercepto
        poly = PolynomialFeatures(degree=g, include_bias=False)
        X_poly = poly.fit_transform(X)

        # Validación cruzada de la regresión lineal sobre las variables polinómicas:
        # return_train_score=True guarda el error sobre los folds de entrenamiento
        # scoring calcula varias métricas a la vez (las de error vienen en negativo porque scikit-learn siempre maximiza)
        scores = cross_validate(LinearRegression(), X_poly, y, cv=cv, return_train_score=True,
                                scoring={"rmse": "neg_root_mean_squared_error",
                                         "mae": "neg_mean_absolute_error",
                                         "r2": "r2"})

        # Guardamos un resumen de este grado: la media de cada métrica en los 5 folds
        # El signo negativo delante de los errores los vuelve a poner en positivo
        resultados.append({
            "grado": g,
            "n_variables": X_poly.shape[1],               # número de columnas tras la transformación polinómica
            "RMSE_train": -scores["train_rmse"].mean(),   # error sobre los datos de train
            "RMSE_cv": -scores["test_rmse"].mean(),       # error sobre los folds no vistos(el que se usa de test)
            "MAE_cv": -scores["test_mae"].mean(),         # error absoluto medio en test (en gramos)
            "R2_cv": scores["test_r2"].mean()             # R cuadrado en test
        })

    # Convertimos la lista de resultados en una tabla con una fila por grado
    return pd.DataFrame(resultados)

# Aplicamos la función solo sobre el conjunto de entrenamiento (el test se reserva para la evaluación final)
resultados = comparar_grados(X_train_scaled, y_train)
resultados

,grado,n_variables,RMSE_train,RMSE_cv,MAE_cv,R2_cv
0,1,20,152.248135,152.272995,120.480379,0.676743
1,2,230,98.057401,98.287982,75.480774,0.865319
2,3,1770,74.889478,76.143098,56.753820,0.919171


El mejor modelo es el de grado 3, basándonos en el RMSE de validación, ya que obtiene el valor más bajo (76,14 €). Además, su MAE es de 56,75 € y su $R^2$  es de 0,919, lo que indica que explica aproximadamente el 91,9 % de la variabilidad de las primas. Por tanto, es el modelo que ofrece mejores resultados de predicción entre los tres modelos comparados.

Y podemos evaluar el conjunto de test con el mejor grado (el que logra menos error en el conjunto de train):

In [9]:

# idxmin() devuelve la fila donde RMSE_cv es mínimo, y .loc[fila, "grado"] extrae el grado de esa fila
# int() lo convierte en un entero normal, porque PolynomialFeatures necesita un número entero
mejor_grado = int(resultados.loc[resultados["RMSE_cv"].idxmin(), "grado"])

# Generamos las variables polinómicas con el grado elegido
poly = PolynomialFeatures(degree=mejor_grado, include_bias=False)
X_train_poly = poly.fit_transform(X_train_scaled)  
X_test_poly = poly.transform(X_test_scaled)        

# Entrenamos el modelo final con todo el conjunto de entrenamiento
modelo_final = LinearRegression()
modelo_final.fit(X_train_poly, y_train)

# Predecimos sobre el test: datos que el modelo no ha visto en ningún momento
pred_test = modelo_final.predict(X_test_poly)

# Evaluamos el modelo una única vez en test: esta es la estimación final de su rendimiento con datos nuevos
print(f"Mejor grado: {mejor_grado}")
print(f"Test:  RMSE = {root_mean_squared_error(y_test, pred_test):.2f} | "   
      f"MAE = {mean_absolute_error(y_test, pred_test):.2f} | "               
      f"R² = {r2_score(y_test, pred_test):.3f}")                             

Mejor grado: 3
Test:  RMSE = 74.97 | MAE = 56.15 | R² = 0.921


In [11]:
# Creamos las variables polinómicas de grado 3
poly = PolynomialFeatures(degree=3, include_bias=False)

X_train_poly = poly.fit_transform(X_train_scaled)

# Entrenamos el modelo con los datos de entrenamiento
modelo_prima = LinearRegression()
modelo_prima.fit(X_train_poly, y_train)

# Preparamos todos los clientes con las mismas variables que en el entrenamiento
X = df.drop(columns=['Prima', 'ID', 'Puntos_Aleman', 'Puntos_Frances',
                     'Unnamed: 0', 'Sistema_Amortizacion'])

# Separamos las variables binarias y numéricas
dummy_cols = [c for c in X.columns if X[c].nunique() == 2]
num_cols = [c for c in X.columns if c not in dummy_cols]

# Utilizamos el mismo escalador del entrenamiento
X_num_scal = pd.DataFrame(
    scaler.transform(X[num_cols]),
    columns=num_cols,
    index=X.index
)

X_scaled = pd.concat([X_num_scal, X[dummy_cols]], axis=1)

# Aplicamos la transformación polinómica a todos los clientes
X_poly = poly.transform(X_scaled)

# Calculamos las primas estimadas
primas_estimadas = modelo_prima.predict(X_poly)

# Creamos un DataFrame con todos los clientes
df_primas = df.copy()
df_primas['Prima_Estimada'] = primas_estimadas

# Mostramos las primas reales y estimadas
df_primas[['ID', 'Prima', 'Prima_Estimada']].head(10)

ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- Unnamed: 0


## 4. Curva de aprendizaje